# Quick box-bowl spot check: reconstruction → activations → steering

Select your **A100 / high-RAM runtime**, then **Run all**. This uses the existing old/new checkpoints and pinned teacher. A healthy comparison environment, local teacher download, and checkpoint copies are reused. The teacher stays loaded while the dictionaries are swapped, avoiding a model reload for every trial. No training or expensive circuit tracing runs here.

**Question:** do the dictionaries reconstruct native MLP outputs, respond to the target instruction, and provide a direction that changes early robot movement?

We use **five unique simulator initial states**, replayed under matched conditions. The actual scene has a bowl **on the cookie box** and another bowl **on the wooden cabinet**. Both native instructions use exactly the same starting physics, pixels, and policy noise. The notebook resolves the benchmark's own descriptions; it does not move or recolor objects.

1. Capture all 18 MLP outputs and transcoder reconstructions for both prompts. Report per-layer relative reconstruction RMSE, sparsity, and action error when all 18 MLPs are replaced together.
2. On starts **0–1**, select up to four sparse features at fixed layer 8 with a consistent box-minus-cabinet response. Freeze the selection separately for each dictionary; feature IDs are not comparable across models.
3. On starts **2–4**, check whether those responses recur. Keep the **cabinet instruction fixed** and compare zero edit, an activity-matched random feature group, and the selected feature group steered toward the box contrast. Edits are capped at 10% of the native MLP output RMS per token. Report realized edit sizes too.
4. Execute only the first predicted action chunk (50 simulator steps), and inspect measured gripper movement and GIFs. This is **up to 48 short condition replays**, not 48 full episodes. No replan or full-task success estimate is attempted.

**Read the controls first:** native prompt switching should produce the expected movement contrast; the zero edit should preserve actions and movement; old/new native teacher actions should agree. A failed control makes a steering result inconclusive. Positive selected-minus-random movement is preliminary evidence only. The score is 50 × change in `(distance to cabinet bowl − distance to box bowl) / bowl separation`, using the initial bowl positions; positive means closer toward the box relative to the native cabinet baseline. It is not success percentage.

The three check starts are excluded from feature selection, but may have appeared in unsupervised transcoder training. Three starts cannot establish significance: the smallest exact two-sided sign-flip p-value is **0.25**. One random group is a control, not a null distribution. This tests a group of features on one task, not a semantic concept or general causal circuit.

**Resume:** `RESUME_RUN_ID="latest"` selects the latest compatible quick run. Every completed trial, activation array, GIF, and shared initial state is verified and committed to Drive. After interruption, Run all skips completed trials and restarts only the unfinished trial. Changed settings/source/checkpoints get a separate run. Setup downloads are cached on the runtime disk; a new Colab VM must recreate its environment and missing downloads. Existing long-comparison outputs are preserved and are not treated as this different experiment.


In [ ]:
# @title Quick test controls — five unique initial states
from pathlib import Path
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908" # @param {type:"string"}
OLD_CHECKPOINT = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"
NEW_CHECKPOINT = "outputs/trajectory-training/pi05-trajectory-transcoders-001/trained_transcoders.pt"
NEW_SHA256 = "ea582a91af113f20c232a545ebc11f47c5e2d6f115929ae7a3a96ccd0f74784e"
CHECKPOINTS = {"old": OLD_CHECKPOINT, "new": NEW_CHECKPOINT}
POLICY_PATH = "lerobot/pi05_libero_finetuned"
SAVE_RUNS_TO_DRIVE = True
MIN_GPU_MEMORY_GB = 20
RESUME_RUN_ID = "latest" # @param {type:"string"}
STEERING_LAYER = 8 # @param {type:"integer"}
SPARSE_FEATURES = 4 # @param {type:"integer"}
RMS_BUDGET = 0.1 # @param {type:"number"}
ACTION_STEPS = 50 # @param {type:"integer"}
SEED = 1000
print("5 unique starts; 2 for feature selection, 3 for exploratory checks.")
print("Up to 48 short condition replays across both dictionaries; one teacher load.")


In [ ]:
# @title Validate Runtime And Mount Drive

import os
import shutil
import subprocess
import time
from pathlib import Path

from google.colab import drive


def gpu_info():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip().splitlines()[0]
        name, mem = [part.strip() for part in out.split(",")]
        return name, int(mem) // 1024
    except Exception:
        return None, 0


gpu_name, gpu_mem_gb = gpu_info()
if gpu_name is None:
    raise RuntimeError("No GPU. Runtime > Change runtime type > L4.")
print(f"GPU: {gpu_name} (~{gpu_mem_gb} GiB)")
if gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
    raise RuntimeError(
        f"{gpu_name} has ~{gpu_mem_gb} GiB; Pi0.5 needs >= {MIN_GPU_MEMORY_GB} GiB. Use an L4 or A100."
    )


def mount_drive_with_retry(mountpoint: str = "/content/drive", attempts: int = 3) -> None:
    """Mount Drive, retrying because `mount failed` is usually transient."""
    if Path(mountpoint, "MyDrive").exists():
        print(f"Drive already mounted at {mountpoint}", flush=True)
        return
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted at {mountpoint} (attempt {attempt})", flush=True)
            return
        except Exception as exc:
            last_error = exc
            print(f"Drive mount attempt {attempt}/{attempts} failed: {exc}", flush=True)
            if attempt < attempts:
                delay = 5 * attempt
                print(f"  retrying in {delay}s", flush=True)
                time.sleep(delay)
    raise RuntimeError(
        f"Could not mount Google Drive after {attempts} attempts (last error: {last_error}).\n"
        "Drive is needed here only to read the transcoder checkpoint.\n"
        "Most likely causes, in order:\n"
        "  1. Other Colab sessions hold Drive mounts. Runtime > Manage sessions, terminate "
        "the ones you are not using, then Runtime > Restart session and rerun.\n"
        "  2. The authorization popup was blocked. Allow popups and third-party cookies.\n"
        "  3. A transient Drive outage. Restart the runtime and retry in a few minutes."
    ) from last_error


mount_drive_with_retry()
DRIVE_ROOT = Path(DRIVE_ROOT)
print("Drive root:", DRIVE_ROOT, "exists:", DRIVE_ROOT.exists())

# Everything this notebook measures is written under DRIVE_ROOT/outputs, because
# Colab's disk is wiped when the runtime ends. If the folder is not actually
# there, copytree would happily create a fresh unshared directory and the run
# would look saved while being invisible to anyone else. Fail here instead.
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
if SAVE_RUNS_TO_DRIVE:
    if not DRIVE_ROOT.exists():
        raise RuntimeError(
            f"{DRIVE_ROOT} does not exist under the mounted Drive.\n"
            "Check DRIVE_ROOT in Controls against the shared folder, and make sure the "
            "folder is added to *your* My Drive (Shared with me is not mounted).\n"
            "Set SAVE_RUNS_TO_DRIVE = False only if you accept losing the run."
        )
    DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
    probe = DRIVE_OUTPUTS / ".write_test"
    probe.write_text("ok")          # a read-only share fails here, not four hours in
    assert probe.read_text() == "ok"
    probe.unlink()
    free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1024**3
    print(f"Drive writable. Free on the mount: {free_gb:.1f} GB")
    print(f"Everything this notebook produces lands under {DRIVE_OUTPUTS}")
else:
    print("SAVE_RUNS_TO_DRIVE is off: this run will be lost when the runtime ends.")


In [ ]:
# @title Install / Repair Runtime (rerun safely after an interrupted install)
import os, subprocess, shlex
from collections import deque
from pathlib import Path

VENV = Path('/content/pi05-comparison-venv')
PYTHON = VENV / 'bin/python'
UV_BIN_DIR = Path('/content/uv-bin')
UV = str(UV_BIN_DIR / 'uv')
INSTALL_LOG = Path('/content/pi05-comparison-install.log')
FORK_COMMIT = 'dcddb970176382c0fcf4521b0c0e6fc15894dfe0'

def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print('$', shlex.join(cmd), flush=True)
    tail = deque(maxlen=100)
    with INSTALL_LOG.open('a') as log:
        log.write('\n$ ' + shlex.join(cmd) + '\n')
        with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            try:
                for line in proc.stdout:
                    print(line, end='', flush=True)
                    log.write(line); log.flush(); tail.append(line)
                status = proc.wait()
            except BaseException:
                proc.terminate()
                try: proc.wait(timeout=5)
                except subprocess.TimeoutExpired: proc.kill(); proc.wait()
                raise
    if status:
        raise RuntimeError(f'Command failed (exit {status}); log: {INSTALL_LOG}\n' + ''.join(tail))
    return subprocess.CompletedProcess(cmd, status)

os.environ['PATH'] = f'{VENV / "bin"}:{UV_BIN_DIR}:' + os.environ['PATH']
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MPLBACKEND'] = 'Agg'

# A Python executable alone does not mean a previous install succeeded.
CHECK = r"""
import json, importlib.metadata as md, importlib.util
expected = {'lerobot':'0.4.4','torch':'2.10.0','torchvision':'0.25.0',
            'hf-libero':'0.1.4','numpy':'2.4.6','matplotlib':'3.10.0','av':'15.1.0'}
for name, value in expected.items():
    assert md.version(name).split('+')[0] == value, (name, md.version(name), value)
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert origin.get('vcs_info',{}).get('commit_id') == 'dcddb970176382c0fcf4521b0c0e6fc15894dfe0'
from transformers.models.siglip import check
assert check.check_whether_transformers_replace_is_installed_correctly()
from lerobot.envs.configs import LiberoEnv
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
import torch, matplotlib, av, numpy
assert torch.cuda.is_available(), 'CUDA unavailable in comparison Python'
assert all(importlib.util.find_spec(n) for n in ('libero','egl_probe'))
print('Validated Pi0.5 fork, simulation imports, CUDA, and versions:', expected)
"""
ready = False
if PYTHON.exists():
    probe = subprocess.run([str(PYTHON), '-c', CHECK], capture_output=True, text=True)
    ready = probe.returncode == 0
    print(probe.stdout if ready else 'Incomplete/incompatible runtime; repairing.\n' + probe.stderr[-2500:])
if not ready:
    apt_env = {**os.environ, 'DEBIAN_FRONTEND':'noninteractive'}
    run(['apt-get','update','-qq'], env=apt_env)
    run(['apt-get','install','-y','-qq','build-essential','ca-certificates','cmake',
         'curl','ffmpeg','git','pkg-config','libegl1','libgl1','libglib2.0-0',
         'libglvnd0','libglx0','libopengl0','libosmesa6-dev','libsm6','libxext6','libxrender1'],env=apt_env)
    if not Path(UV).exists():
        UV_BIN_DIR.mkdir(parents=True,exist_ok=True)
        run(['curl','-LsSf','https://astral.sh/uv/install.sh','-o','/tmp/install-uv.sh'])
        run(['sh','/tmp/install-uv.sh'], env={**os.environ,'UV_INSTALL_DIR':str(UV_BIN_DIR)})
    if not PYTHON.exists():
        run([UV,'python','install','3.12'])
        run([UV,'venv',str(VENV),'--python','3.12'])
    # Match training: egl-probe requires pre-CMake-4 build tooling.
    run([UV,'pip','install','--python',PYTHON,'cmake==3.31.6','setuptools>=71,<81','wheel'])
    run([UV,'pip','install','--python',PYTHON,'--no-build-isolation-package','egl-probe','egl-probe==1.0.2'])
    override = Path('/content/pi05-comparison-transformers-overrides.txt')
    override.write_text(f'transformers @ git+https://github.com/huggingface/transformers.git@{FORK_COMMIT}\n')
    run([UV,'pip','install','--python',PYTHON,'--torch-backend','cu128',
         '--overrides',override,'--no-build-isolation-package','egl-probe','cmake==3.31.6',
         'lerobot[libero]==0.4.4','torch==2.10.0','torchvision==0.25.0',
         'hf-libero==0.1.4','hf-transfer','opencv-python','numpy==2.4.6',
         'matplotlib==3.10.0','av==15.1.0','scikit-learn','joblib','pillow'])
run([PYTHON,'-c',CHECK])
print('Runtime ready. All model stages run in this separate Python; no kernel restart needed.')


In [ ]:
# @title Restore the bundled source (no GitHub update required)
import base64, hashlib, json, os, subprocess, zlib
from pathlib import Path
SOURCE_SHA256 = 'addf6c6777cd35828bd90628e22a6f3267c421c51e3f21cde4d9061f3f8f0723'
RUNNER_SHA256 = '9f11f91917c28cf8f5db49190e8e2efbac93f10ec3328fe0526823dda3db6b0a'
SOURCE_BASE_COMMIT = '79ccf32a10df551a0d33167d43e03cee85241f18'
payload = '...'
raw = zlib.decompress(base64.b64decode(payload))
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256
LOCAL_REPO = Path("/content/pi05-quick-box-source") / SOURCE_SHA256[:12]
for name, contents in json.loads(raw).items():
    target = LOCAL_REPO / name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(contents)
os.environ["PYTHONPATH"] = os.pathsep.join([str(LOCAL_REPO/'src'),str(LOCAL_REPO/'scripts')])
print("Source:",LOCAL_REPO)


In [ ]:
# @title Check evaluation imports before running either model
preflight_code = """
import importlib, sys
from pi05_mi.lerobot_runtime import patch_transformers_causal_mask_compat, patch_pi05_checkpoint_key_compat
patch_transformers_causal_mask_compat()
patch_pi05_checkpoint_key_compat()
for name in ('run_pi05_quick_box', 'probe_pi05_transcoder_counterfactual', 'compare_transcoder_fidelity',
             'collect_pi05_transcoder_features', 'trace_pi05_transcoder_circuit'):
    importlib.import_module(name)
    print('Import OK:', name, flush=True)
assert 'train_pi05_transcoders' not in sys.modules
import torch, numpy as np
from probe_pi05_transcoder_counterfactual import LatentAccumulator
gpu=torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name}; VRAM: {gpu.total_memory / 2**30:.1f} GiB', flush=True)
# Check the optimized transport on the actual GPU before reusing old results.
reference=LatentAccumulator(defer_transfers=False)
fast=LatentAccumulator()
reference.enabled=fast.enabled=True
values=torch.arange(1024, device='cuda', dtype=torch.float32).reshape(1,8,128).to(torch.bfloat16)
for step in range(3):
    reference.callback('test',0,values+step,torch.tensor([0.5],device='cuda'))
    fast.callback('test',0,values+step,torch.tensor([0.5],device='cuda'))
fast.flush()
for key in reference.mean:
    np.testing.assert_array_equal(reference.mean[key],fast.mean[key])
    np.testing.assert_array_equal(reference.max[key],fast.max[key])
print('Batched GPU statistics transfer verified; experiment settings unchanged.', flush=True)
print('Evaluation import preflight passed.', flush=True)
"""
run([PYTHON, '-u', '-c', preflight_code])


In [ ]:
# @title HF Token And LIBERO Assets

import os
from pathlib import Path

token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
if not token:
    secret_file = DRIVE_ROOT / "secrets/HF_TOKEN.txt"
    if secret_file.exists():
        token = secret_file.read_text().strip()
if not token:
    raise RuntimeError(
        "No Hugging Face token. Add HF_TOKEN as a Colab secret (key icon in the sidebar), "
        f"or place it at {DRIVE_ROOT / 'secrets/HF_TOKEN.txt'}. "
        f"{POLICY_PATH} is gated, so the download needs it."
    )
os.environ["HF_TOKEN"] = token
print("HF token loaded.")

# LIBERO ships without its scene assets; fetch them into the installed package.
assets_code = r"""
import shutil, site, os
from pathlib import Path
from huggingface_hub import snapshot_download

roots = [Path(p) for p in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
libero_root = next((r / "libero" / "libero" for r in roots if (r / "libero" / "libero").exists()), None)
if libero_root is None:
    raise RuntimeError("Installed LIBERO package not found")
assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
if required.exists():
    print("LIBERO assets already present:", required)
else:
    print("Downloading LIBERO assets (cached files are reused)...", flush=True)
    snap = Path(snapshot_download(repo_id="lerobot/libero-assets", repo_type="dataset",
                                  token=os.environ.get("HF_TOKEN") or None))
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snap.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
    if not required.exists():
        raise FileNotFoundError(f"LIBERO asset install failed; missing {required}")
    print("LIBERO assets installed:", required)
"""
run([str(PYTHON), "-u", "-c", assets_code])

# Freeze the teacher to the exact revision recorded by the new training run.
teacher_result = LOCAL_REPO / 'teacher_path.txt'
run([PYTHON, '-u', LOCAL_REPO / 'scripts/download_pi05_comparison_teacher.py',
     '--result-path', teacher_result])
POLICY_PATH = teacher_result.read_text().strip()
print('Pinned teacher:', POLICY_PATH)
# Parse the actual checkpoint config before launching any evaluation tasks.
run([PYTHON, '-u', '-c',
     "from argparse import Namespace; from pi05_mi.lerobot_runtime import _load_train_config; "
     "import sys; cfg=_load_train_config(Namespace(policy_path=sys.argv[1],local_files_only=True)); "
     "print('Teacher config OK:', type(cfg.policy).__name__)", POLICY_PATH])


In [ ]:
# @title Verify and stage both checkpoints
import hashlib, shutil
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(16*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
CHECKPOINT_HASHES, LOCAL_CHECKPOINTS = {}, {}
for arm,relative in CHECKPOINTS.items():
    source=DRIVE_ROOT/relative
    if not source.is_file(): raise FileNotFoundError(source)
    local=Path('/content/pi05-comparison-checkpoints')/arm/source.name
    local.parent.mkdir(parents=True,exist_ok=True)
    print('Hashing',arm,'checkpoint...',flush=True)
    expected=sha256(source)
    if arm=='new' and expected!=NEW_SHA256:
        raise ValueError('New checkpoint does not match the completed, downloaded training bundle')
    if not local.exists() or sha256(local)!=expected:
        partial=local.with_suffix('.partial');shutil.copyfile(source,partial)
        if sha256(partial)!=expected: raise IOError('Checkpoint copy hash mismatch')
        partial.replace(local)
    CHECKPOINT_HASHES[arm]=expected;LOCAL_CHECKPOINTS[arm]=str(local)
    print(arm,local,expected)
run([PYTHON,'-c',"from pi05_mi.transcoder_checkpoints import load_transcoders; import sys; "
     "m=load_transcoders(sys.argv[1],layers=[0]); print(type(next(iter(m.values()))).__name__)",
     LOCAL_CHECKPOINTS['old']])
run([PYTHON,'-c',"from pi05_mi.transcoder_checkpoints import load_transcoders; import sys; "
     "m=load_transcoders(sys.argv[1],layers=[0]); print(type(next(iter(m.values()))).__name__)",
     LOCAL_CHECKPOINTS['new']])


In [ ]:
# @title Select a compatible run and restore completed trials
import sys
sys.path.insert(0,str(LOCAL_REPO/'scripts'))
from comparison_resume import ResumeCache, select_run, atomic_json
identity = dict(protocol="quick-box-v1", source=SOURCE_SHA256, runner=RUNNER_SHA256,
    checkpoints=CHECKPOINT_HASHES, teacher_revision="8e174154ef5f6c60a8da12ae99c303d8963138c1",
    task="pick_up_the_black_bowl_on_the_cookie_box_and_place_it_on_the_plate",
    starts=[0,1,2,3,4], discovery=[0,1], spot_check=[2,3,4], seed=SEED,
    layer=STEERING_LAYER, features=SPARSE_FEATURES, budget=RMS_BUDGET,
    action_steps=ACTION_STEPS, inference_steps=10, policy_dtype="bfloat16")
local_parent = LOCAL_REPO/'outputs/quick-box'
drive_parent = DRIVE_ROOT/'outputs/quick-box' if SAVE_RUNS_TO_DRIVE else None
RUN_ID = select_run(local_parent,drive_parent,identity,RESUME_RUN_ID)
QUICK_ROOT = local_parent/RUN_ID
QUICK_DRIVE = drive_parent/RUN_ID if drive_parent else None
CACHE = ResumeCache(QUICK_ROOT,QUICK_DRIVE,LOCAL_REPO)
(QUICK_ROOT/'source_snapshot.json').write_bytes(raw)
atomic_json(QUICK_ROOT/'source_manifest.json',dict(sha256=SOURCE_SHA256,base_commit=SOURCE_BASE_COMMIT))
CACHE.complete('source',CACHE.signature(['source']),
    [QUICK_ROOT/'source_snapshot.json',QUICK_ROOT/'source_manifest.json'])
print("Run:",RUN_ID,"\nResults:",QUICK_DRIVE or QUICK_ROOT)
print("Rerun this notebook with latest to reuse verified completed trials.")


In [ ]:
# @title Run reconstruction, feature discovery, and controlled steering
cmd=[PYTHON,'-u',LOCAL_REPO/'scripts/run_pi05_quick_box.py',
     '--old-checkpoint',LOCAL_CHECKPOINTS['old'],'--new-checkpoint',LOCAL_CHECKPOINTS['new'],
     '--policy-path',POLICY_PATH,'--resume-root',QUICK_ROOT,
     '--layer',STEERING_LAYER,'--features',SPARSE_FEATURES,'--budget',RMS_BUDGET,
     '--action-steps',ACTION_STEPS,'--seed',SEED]
if QUICK_DRIVE:cmd+=['--drive-root',QUICK_DRIVE]
run(cmd)


In [ ]:
# @title Read the preliminary results
import json, csv
from IPython.display import display, Image, HTML
report=json.loads((QUICK_ROOT/'quick_report.json').read_text())
print(report['control'])
for arm,summary in report['models'].items():
    print("\n",arm.upper(),"checks:",summary['checks'])
    print("Selection:",summary['selection'].get('sparse',summary['selection']))
    for row in summary['movement']:
        print({k:round(v,4) if isinstance(v,float) else v for k,v in row.items() if not isinstance(v,dict)})
    print(summary['interpretation'])
print("\nNative teacher repeat errors (should be zero or negligible):")
for row in report['paired_native']:print(row['start'],row['native_action_error']['full'])
for name in ('reconstruction','activations','movement'):
    display(Image(filename=str(QUICK_ROOT/'figures'/f'{name}.png')))
print("CSV tables and PNG figures:", (QUICK_DRIVE or QUICK_ROOT)/'figures')
print("No statistical significance claim: with 3 paired starts, the smallest two-sided sign-flip p is 0.25.")


In [ ]:
# @title Watch the first held-out start — matched starting images
from IPython.display import display, Image, HTML
import html
case=f"task{report['task_id']}_init2"
for arm in ('old','new'):
    display(HTML(f'<h3>{arm}</h3>'))
    for name,label in [('native_-1','Cabinet instruction'),('native_1','Box instruction'),
                       ('random','Cabinet instruction + random edit'),('steer','Cabinet instruction + selected edit')]:
        path=QUICK_ROOT/arm/'cases'/case/name/'preview.gif'
        if path.exists():
            display(HTML('<b>'+html.escape(label)+'</b>'))
            display(Image(filename=str(path)))
print('GIFs sample every fifth simulator step; playback is illustrative, not a timing measurement.')
